# Chapter 4 — Classification & Regression: **from scratch AND with Keras**

Maps to Chollet Ch.4 (binary / multiclass / regression). Your course also demands **from-scratch**
ability (Perceptron, MLP backprop, Kohonen). So this notebook is split:

- **Part A — From scratch** (NumPy only): Perceptron, why XOR needs hidden layers, and a full
  2-layer **MLP with backprop** in your course's notation (`W_ij`, `W_jk`, thresholds, sigmoid gain `k`, `eta`).
- **Part B — With Keras**: the same three task types in a few lines, plus K-fold cross-validation.

### The decision table (memorize — picking these wrong = silent failure)
| task | output meaning | last layer | loss | metric |
|---|---|---|---|---|
| **binary** | P(class 1) | `Dense(1, "sigmoid")` | `binary_crossentropy` | accuracy |
| **multiclass** (int labels) | P(each class) | `Dense(C, "softmax")` | `sparse_categorical_crossentropy` | accuracy |
| **multiclass** (one-hot) | P(each class) | `Dense(C, "softmax")` | `categorical_crossentropy` | accuracy |
| **multilabel** | P(each label, indep.) | `Dense(C, "sigmoid")` | `binary_crossentropy` | — |
| **regression** | a number | `Dense(1)` *(no activation)* | `mse` | `mae` |


## Data loaders (uses your lab CSVs; falls back if absent)
Put `diabetes.csv` (Pima, binary) and `Iris.csv` next to this notebook. Both auto-fallback.

In [ ]:
import numpy as np, pandas as pd, os, urllib.request

def load_diabetes_binary():
    # Pima: 8 features + Outcome(0/1). Try local lab CSV (has header), else download headerless.
    if os.path.exists("diabetes.csv"):
        df = pd.read_csv("diabetes.csv")
        y = df.iloc[:, -1].astype(int).values
        X = df.iloc[:, :-1].astype(float).values
    else:
        os.makedirs("data", exist_ok=True); p="data/pima.csv"
        if not os.path.exists(p):
            urllib.request.urlretrieve(
                "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv", p)
        a=np.loadtxt(p, delimiter=","); X,y=a[:,:-1], a[:,-1].astype(int)
    return X, y

def load_iris_multiclass():
    from sklearn.preprocessing import LabelEncoder
    if os.path.exists("Iris.csv"):
        df = pd.read_csv("Iris.csv")
        if "Id" in df.columns: df = df.drop(columns=["Id"])
        X = df.iloc[:, :-1].astype(float).values
        y = LabelEncoder().fit_transform(df.iloc[:, -1].values)
    else:
        from sklearn.datasets import load_iris; d=load_iris(); X,y=d.data, d.target
    return X, y

Xb, yb = load_diabetes_binary(); print("diabetes:", Xb.shape, "classes:", np.unique(yb))
Xi, yi = load_iris_multiclass(); print("iris:", Xi.shape, "classes:", np.unique(yi))


---
# Part A — From scratch (NumPy only)

## A1. The Perceptron (single neuron)
A perceptron computes `y = step(w·x + bias)`. Trick: append a constant **1** to every input so the
**bias becomes a normal weight** (this replaces the fixed threshold `θ`; your lab's `θ=0.5` is just `-bias`).

**Learning rule (batch):** `w ← w + η · (y_true − y_pred)ᵀ · X`. Repeat until no errors (if data is
linearly separable, it's *guaranteed* to converge — Perceptron Convergence Theorem).

In [ ]:
import numpy as np
def perceptron_fit(X, y, lr=1.0, max_epochs=100):
    Xb = np.c_[X, np.ones(len(X))]          # add bias column
    w  = np.zeros(Xb.shape[1])
    for ep in range(max_epochs):
        y_pred = (Xb @ w >= 0).astype(int)  # step activation (threshold 0 after bias trick)
        if np.array_equal(y_pred, y):
            return w, ep, True
        w = w + lr * ((y - y_pred) @ Xb)    # batch update
    return w, max_epochs, False

for name, y in [("AND", np.array([0,0,0,1])),
                ("OR",  np.array([0,1,1,1])),
                ("XOR", np.array([0,1,1,0]))]:
    X = np.array([[0,0],[0,1],[1,0],[1,1]])
    w, ep, ok = perceptron_fit(X, y)
    print(f"{name:3s}  converged={ok}  epochs={ep}")


## A2. Why XOR fails → we need hidden layers + a nonlinearity
A single perceptron is a **line**; XOR isn't linearly separable, so it never converges. Stacking *linear*
layers doesn't help (linear∘linear = linear). The fix: **hidden layer + nonlinear activation** (sigmoid/relu).
That's the MLP.

## A3. Two-layer MLP with backprop — *your course's notation*
- indices: **i**=input, **j**=hidden, **k**=output. Weights `W_ij` (in→hidden), `W_jk` (hidden→out).
- thresholds `U_hj`, `U_ok`. We use **bias = −U** so `net = X·W + b` (cleaner signs; identical model).
- sigmoid with **gain** `k`: `σ(net) = 1/(1+e^(−k·net))`,  `σ'(net) = k·σ·(1−σ)`.

**Forward**
```
NETh = X·W_ij + b_h ;  H = σ(k1·NETh)
NETo = H·W_jk + b_o ;  O = σ(k2·NETo)
```
**Backprop (MSE loss E = ½Σ(T−O)²)**
```
δ_o = (O − T) · k2·O·(1−O)          # output error signal
δ_h = (δ_o · W_jkᵀ) · k1·H·(1−H)    # propagate back
grad W_jk = Hᵀ·δ_o ,  grad b_o = Σδ_o
grad W_ij = Xᵀ·δ_h ,  grad b_h = Σδ_h
W ← W − η·gradW ,  b ← b − η·grad b
```

In [ ]:
import numpy as np
def sigmoid(net, k=1.0): return 1.0/(1.0+np.exp(-k*net))

class ScratchMLP:
    def __init__(self, n_in, n_hidden, n_out, eta=0.5, k1=1.0, k2=1.0, seed=42):
        r = np.random.default_rng(seed)
        self.W_ij = r.normal(0, 0.3, (n_in, n_hidden));  self.b_h = np.zeros(n_hidden)
        self.W_jk = r.normal(0, 0.3, (n_hidden, n_out));  self.b_o = np.zeros(n_out)
        self.eta, self.k1, self.k2 = eta, k1, k2
    def forward(self, X):
        self.H = sigmoid(X @ self.W_ij + self.b_h, self.k1)
        self.O = sigmoid(self.H @ self.W_jk + self.b_o, self.k2)
        return self.O
    def train(self, X, T, epochs=2000, verbose=False):
        n = len(X)
        for ep in range(epochs):
            O = self.forward(X)
            dO = (O - T) * self.k2 * O * (1 - O)
            dH = (dO @ self.W_jk.T) * self.k1 * self.H * (1 - self.H)
            self.W_jk -= self.eta * (self.H.T @ dO) / n; self.b_o -= self.eta * dO.mean(0)
            self.W_ij -= self.eta * (X.T  @ dH) / n;     self.b_h -= self.eta * dH.mean(0)
            if verbose and ep % 500 == 0:
                print(f"  epoch {ep:4d}  MSE={np.mean((T-O)**2):.4f}")
        return self


### A3 (run) — binary classification on diabetes, from scratch

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.2, random_state=42, stratify=yb)
sc = StandardScaler().fit(Xtr); Xtr_s = sc.transform(Xtr); Xte_s = sc.transform(Xte)
T_tr = ytr.reshape(-1, 1).astype(float)               # binary -> 1 output node

net = ScratchMLP(n_in=Xtr_s.shape[1], n_hidden=8, n_out=1, eta=0.5).train(Xtr_s, T_tr, epochs=3000, verbose=True)
acc = ((net.forward(Xte_s) > 0.5).astype(int).ravel() == yte).mean()
print(f"\nFROM-SCRATCH binary test accuracy = {acc:.3f}")


## A4. Multiclass from scratch (Iris) — one-hot targets, argmax to predict

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(Xi, yi, test_size=0.2, random_state=42, stratify=yi)
sc = StandardScaler().fit(Xtr); Xtr_s = sc.transform(Xtr); Xte_s = sc.transform(Xte)
C = len(np.unique(yi))
T_tr = np.eye(C)[ytr]                                  # one-hot: class 2 -> [0,0,1]

net = ScratchMLP(n_in=Xtr_s.shape[1], n_hidden=10, n_out=C, eta=0.5).train(Xtr_s, T_tr, epochs=3000)
pred = net.forward(Xte_s).argmax(1)                    # highest output node = predicted class
print(f"FROM-SCRATCH multiclass test accuracy = {(pred == yte).mean():.3f}")


---
# Part B — With Keras (the production way)
Same three tasks, ~5 lines each. Notice how the **decision table** maps directly to `loss` + last layer.

In [ ]:
import os; os.environ["KERAS_BACKEND"]="tensorflow"
import keras
from keras import layers
import numpy as np


## B1. Binary classification (diabetes) — sigmoid + binary_crossentropy

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.2, random_state=42, stratify=yb)
sc = StandardScaler().fit(Xtr); Xtr=sc.transform(Xtr); Xte=sc.transform(Xte)

model = keras.Sequential([
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1,  activation="sigmoid"),     # P(diabetes)
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(Xtr, ytr, epochs=50, batch_size=16, validation_split=0.2, verbose=0)
print("binary test:", model.evaluate(Xte, yte, verbose=0, return_dict=True))


## B2. Multiclass (Iris) — softmax + sparse_categorical_crossentropy (integer labels)

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(Xi, yi, test_size=0.2, random_state=42, stratify=yi)
sc = StandardScaler().fit(Xtr); Xtr=sc.transform(Xtr); Xte=sc.transform(Xte)
C = len(np.unique(yi))

model = keras.Sequential([
    layers.Dense(16, activation="relu"),
    layers.Dense(C,  activation="softmax"),     # C probabilities summing to 1
])
# integer labels -> 'sparse_...'. If you one-hot the labels, use 'categorical_crossentropy' instead.
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(Xtr, ytr, epochs=100, batch_size=8, validation_split=0.2, verbose=0)
print("multiclass test:", model.evaluate(Xte, yte, verbose=0, return_dict=True))


## B3. Regression (housing) — **no final activation**, mse loss, mae metric
Key differences from classification: last layer is a bare `Dense(1)`, loss is `mse`, and you watch `mae`
(mean absolute error, in the target's units). **Always normalize features** for regression.

In [ ]:
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing()
Xr, yr = data.data, data.target                  # predict median house value
Xtr, Xte, ytr, yte = train_test_split(Xr, yr, test_size=0.2, random_state=42)
sc = StandardScaler().fit(Xtr); Xtr=sc.transform(Xtr); Xte=sc.transform(Xte)

model = keras.Sequential([
    layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(1),                              # NO activation for regression
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.fit(Xtr, ytr, epochs=30, batch_size=64, validation_split=0.2, verbose=0)
res = model.evaluate(Xte, yte, verbose=0, return_dict=True)
print(f"regression test: mae={res['mae']:.3f} (in $100k units)")


## B4. K-fold cross-validation — for **small** datasets (Chollet's housing technique)
With few samples, a single train/val split is noisy. K-fold trains K times on different folds and
averages the scores — a far more reliable estimate. Use it whenever data is scarce (most lab tasks!).

In [ ]:
from sklearn.model_selection import KFold

def build():
    m = keras.Sequential([layers.Dense(32, activation="relu"),
                          layers.Dense(1, activation="sigmoid")])
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return m

X_all = StandardScaler().fit_transform(Xb)        # diabetes again
scores = []
for tr_idx, va_idx in KFold(n_splits=5, shuffle=True, random_state=42).split(X_all):
    m = build()
    m.fit(X_all[tr_idx], yb[tr_idx], epochs=40, batch_size=16, verbose=0)
    scores.append(m.evaluate(X_all[va_idx], yb[va_idx], verbose=0, return_dict=True)["accuracy"])
print("5-fold accuracies:", [round(s,3) for s in scores])
print(f"mean = {np.mean(scores):.3f}  ±  {np.std(scores):.3f}")


## B5. Validation set, overfitting & the *information leak*
- Train ↘ but **val** loss starts ↗ → **overfitting**. Use the val curve to pick #epochs / model size.
- Never tune on the **test** set. Every decision you make based on a set "leaks" info into the model;
  the test set must stay untouched until the final, one-time evaluation.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.2, random_state=42, stratify=yb)
sc = StandardScaler().fit(Xtr); Xtr=sc.transform(Xtr); Xte=sc.transform(Xte)
m = keras.Sequential([layers.Dense(64,activation="relu"),
                      layers.Dense(64,activation="relu"),
                      layers.Dense(1,activation="sigmoid")])
m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
h = m.fit(Xtr, ytr, epochs=200, batch_size=16, validation_split=0.2, verbose=0)

import matplotlib.pyplot as plt
plt.plot(h.history["loss"], label="train"); plt.plot(h.history["val_loss"], label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("watch val loss turn upward = overfitting"); plt.show()


---
# ✍️ PROBLEMS

### P1 — Perceptron from scratch
Modify `perceptron_fit` to also **return the number of misclassified points each epoch**, and plot that
count vs epoch for OR. Then feed it a 2D linearly-separable random blob dataset and draw the final
decision line `w0·x + w1·y + b = 0`.

In [ ]:
# TODO


### P2 — Extend the from-scratch MLP
Add an `accuracy()` method and a `predict()` method to `ScratchMLP`. Then train it on Iris **with one-hot
targets** and report a confusion matrix (you may use `sklearn.metrics.confusion_matrix`). Try `n_hidden ∈ {3,10,30}`
— how does test accuracy change?

In [ ]:
# TODO


### P3 — ReLU hidden units from scratch
Replace the hidden-layer sigmoid in `ScratchMLP` with **ReLU** (`max(0,net)`, derivative `1 if net>0 else 0`).
Keep sigmoid on the output. Retrain on diabetes. Does it train faster / better? (Watch for "dead" units.)

In [ ]:
# TODO


### P4 — Keras, all three tasks, your way
Without copying B1–B3: build, compile, and evaluate a Keras model for (a) diabetes binary, (b) Iris multiclass,
(c) California regression. For each, **justify your last layer + loss using the decision table** in a comment.
Beat a majority-class baseline (classification) / mean-predictor baseline (regression).

In [ ]:
# TODO


---
# 📋 TEMPLATES

### T1 — Perceptron from scratch

In [ ]:
import numpy as np
def perceptron_fit(X, y, lr=1.0, max_epochs=100):
    Xb = np.c_[X, np.ones(len(X))]; w = np.zeros(Xb.shape[1])
    for ep in range(max_epochs):
        yp = (Xb @ w >= 0).astype(int)
        if np.array_equal(yp, y): return w, ep, True
        w = w + lr * ((y - yp) @ Xb)
    return w, max_epochs, False


### T2 — 2-layer MLP from scratch (course notation; sigmoid, gain k, eta)

In [ ]:
import numpy as np
def sigmoid(net, k=1.0): return 1.0/(1.0+np.exp(-k*net))
class ScratchMLP:
    def __init__(self, n_in, n_hidden, n_out, eta=0.5, k1=1.0, k2=1.0, seed=42):
        r=np.random.default_rng(seed)
        self.W_ij=r.normal(0,.3,(n_in,n_hidden)); self.b_h=np.zeros(n_hidden)
        self.W_jk=r.normal(0,.3,(n_hidden,n_out)); self.b_o=np.zeros(n_out)
        self.eta,self.k1,self.k2=eta,k1,k2
    def forward(self,X):
        self.H=sigmoid(X@self.W_ij+self.b_h,self.k1)
        self.O=sigmoid(self.H@self.W_jk+self.b_o,self.k2); return self.O
    def train(self,X,T,epochs=2000):
        n=len(X)
        for _ in range(epochs):
            O=self.forward(X)
            dO=(O-T)*self.k2*O*(1-O)
            dH=(dO@self.W_jk.T)*self.k1*self.H*(1-self.H)
            self.W_jk-=self.eta*(self.H.T@dO)/n; self.b_o-=self.eta*dO.mean(0)
            self.W_ij-=self.eta*(X.T@dH)/n;      self.b_h-=self.eta*dH.mean(0)
        return self
# binary: n_out=1, T=y.reshape(-1,1), predict (O>0.5)
# multiclass: n_out=C, T=np.eye(C)[y], predict O.argmax(1)


### T3 — Keras by task type (fill the marked spots)

In [ ]:
import os; os.environ["KERAS_BACKEND"]="tensorflow"
import keras; from keras import layers

# BINARY
m = keras.Sequential([layers.Dense(16,activation="relu"),
                      layers.Dense(1,activation="sigmoid")])
m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# MULTICLASS (integer labels)
m = keras.Sequential([layers.Dense(16,activation="relu"),
                      layers.Dense(NUM_CLASSES,activation="softmax")])
m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# REGRESSION
m = keras.Sequential([layers.Dense(64,activation="relu"),
                      layers.Dense(64,activation="relu"),
                      layers.Dense(1)])
m.compile(optimizer="adam", loss="mse", metrics=["mae"])


### T4 — Always-do preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
Xtr,Xte,ytr,yte = train_test_split(X, y, test_size=0.2, random_state=42,
                                   stratify=y)        # drop stratify for regression
sc = StandardScaler().fit(Xtr)                        # FIT ON TRAIN ONLY
Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)


### T5 — K-fold cross-validation

In [ ]:
from sklearn.model_selection import KFold
import numpy as np
scores=[]
for tr,va in KFold(5, shuffle=True, random_state=42).split(X):
    m = build_model()                 # returns a fresh compiled model
    m.fit(X[tr], y[tr], epochs=40, batch_size=16, verbose=0)
    scores.append(m.evaluate(X[va], y[va], verbose=0, return_dict=True)["accuracy"])
print(np.mean(scores), "+/-", np.std(scores))


---
### ✅ Checklist
- [ ] Pick last-layer + loss from the decision table for any task, instantly.
- [ ] Code a perceptron from scratch; explain the bias-trick and why XOR fails.
- [ ] Derive & code 2-layer backprop in course notation (δ_o, δ_h, weight/bias updates).
- [ ] Train the scratch MLP on binary (diabetes) and multiclass (Iris) to good accuracy.
- [ ] Do all three tasks in Keras; normalize features; read overfitting from val curves.
- [ ] Run K-fold CV and report mean ± std.

**Next: Chapter 5** — *Fundamentals of ML*: overfitting/underfitting, regularization (dropout, L2, early
stopping), and how to systematically push accuracy up. (Kohonen/SOM from scratch + CNN-from-scratch get
their own dedicated notebooks when we hit Ch.8.) Say "Chapter 5".